# 🎯 MMR: Maximal Marginal Relevance for Diverse Retrieval

**Course Reference:** [Ultimate RAG Bootcamp Using Langchain, LangGraph & Langsmith](https://www.udemy.com/course/ultimate-rag-bootcamp-using-langchainlanggraph-langsmith)

---

## 📚 Learning Objectives

By the end of this notebook, you will understand:
1. What **Maximal Marginal Relevance (MMR)** is and why it matters
2. How MMR balances **relevance** and **diversity** in retrieval
3. How to implement MMR retrieval with LangChain and FAISS
4. When to use MMR vs. standard similarity search

---

## 🧠 Key Concepts

### What is MMR?

MMR is a retrieval technique that optimizes for **both relevance AND diversity**. It prevents returning redundant, near-duplicate documents.

### The MMR Formula

```
MMR = λ × Relevance(doc, query) - (1-λ) × max(Similarity(doc, selected_docs))
```

Where:
- **λ (lambda)**: Controls the trade-off (0.0 = max diversity, 1.0 = max relevance)
- **Relevance**: How similar the document is to the query
- **Similarity to selected**: How similar the document is to already-selected documents

### Why Use MMR?

| Problem with Standard Retrieval | MMR Solution |
|--------------------------------|--------------|
| Returns near-duplicate chunks | Penalizes redundant documents |
| Wastes context window space | Maximizes information per document |
| May miss diverse perspectives | Encourages varied content |

---

## 🔍 Understanding MMR in Practice

**MMR (Maximal Marginal Relevance)** is a powerful diversity-aware retrieval technique used in information retrieval and RAG pipelines to balance relevance and novelty when selecting documents.

### How MMR Works Step-by-Step:

1. **Initial Candidate Selection**: Retrieve a larger set of potentially relevant documents
2. **First Selection**: Pick the most relevant document to the query
3. **Iterative Selection**: For each subsequent document:
   - Score by relevance to query (higher = better)
   - Penalize by similarity to already-selected documents (higher similarity = lower score)
   - Select the document with the best combined score
4. **Result**: A set of documents that are both relevant AND diverse

### Visual Representation:

```
Query: "How to build AI applications?"

Standard Similarity:           MMR Retrieval:
─────────────────────          ─────────────────────
1. Building AI apps ✓          1. Building AI apps ✓
2. Creating AI apps ✗ (similar) 2. LLM frameworks ✓ (diverse)
3. Making AI apps ✗ (similar)   3. Deployment tips ✓ (diverse)
4. AI app development ✗        4. Best practices ✓ (diverse)
```

In [ ]:
# ============================================================================
# STEP 1: IMPORT REQUIRED LIBRARIES
# ============================================================================

# FAISS: Facebook AI Similarity Search - supports MMR out of the box
from langchain_community.vectorstores import FAISS

# HuggingFaceEmbeddings: Converts text to dense vectors for similarity search
from langchain_huggingface import HuggingFaceEmbeddings

# TextLoader: Loads plain text files into LangChain Document format
from langchain.document_loaders import TextLoader

# RecursiveCharacterTextSplitter: Splits documents into chunks intelligently
from langchain.text_splitter import RecursiveCharacterTextSplitter

# init_chat_model: Universal initializer for chat models (OpenAI, Groq, etc.)
from langchain.chat_models import init_chat_model

# PromptTemplate: Creates reusable prompt templates
from langchain.prompts import PromptTemplate

# create_stuff_documents_chain: Combines documents into LLM context
from langchain.chains.combine_documents import create_stuff_documents_chain

# create_retrieval_chain: Creates a full RAG chain with retrieval
from langchain.chains.retrieval import create_retrieval_chain

print("✅ All libraries imported successfully!")

In [ ]:
# ============================================================================
# STEP 2: SETUP ENVIRONMENT AND API KEYS
# ============================================================================
# Load environment variables from .env file for secure API key management

import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Set API keys for different services
# - OpenAI: For embeddings (optional, we'll use HuggingFace instead)
# - Groq: For fast, free LLM inference
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

print("✅ Environment configured!")
print("   - OpenAI API key: loaded")
print("   - Groq API key: loaded")


In [ ]:
# ============================================================================
# STEP 3: LOAD AND CHUNK THE DOCUMENT
# ============================================================================
# Load the document and split it into smaller chunks for retrieval
# Smaller chunks (300 chars) work well for MMR to show diversity

# Load the text file containing LangChain RAG information
loader = TextLoader("langchain_rag_dataset.txt")
raw_docs = loader.load()

print(f"📄 Loaded {len(raw_docs)} document(s)")
print(f"   Total characters: {len(raw_docs[0].page_content)}")

# Split into smaller chunks for better retrieval granularity
# - chunk_size=300: Smaller chunks make it easier to see MMR's diversity effect
# - chunk_overlap=50: Overlap preserves context across chunk boundaries
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)

print(f"\n✅ Split into {len(chunks)} chunks")
print(f"\n📋 Sample chunks:")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n[Chunk {i+1}]: {chunk.page_content[:100]}...")

chunks

In [ ]:
# ============================================================================
# STEP 4: CREATE FAISS VECTOR STORE
# ============================================================================
# Create a vector store using FAISS, which supports MMR natively
# FAISS stores document embeddings for efficient similarity search

# Initialize the embedding model
# "all-MiniLM-L6-v2" is a lightweight, efficient model:
# - 384-dimensional embeddings
# - Fast inference (CPU-friendly)
# - Good quality for general semantic similarity
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create FAISS vector store from document chunks
# This embeds all chunks and stores them in a FAISS index
vectorstore = FAISS.from_documents(chunks, embedding_model)

print("✅ FAISS vector store created!")
print(f"   - Embedding model: all-MiniLM-L6-v2 (384-dim)")
print(f"   - Documents indexed: {len(chunks)}")
print(f"   - Vector store supports: similarity_search, MMR, similarity_search_with_score")

In [ ]:
# ============================================================================
# STEP 5: CREATE MMR RETRIEVER 🎯
# ============================================================================
# This is the key step! We create a retriever that uses MMR instead of 
# standard similarity search.

# Create MMR retriever with specific parameters:
# - search_type="mmr": Use Maximal Marginal Relevance instead of similarity
# - k=3: Return 3 diverse, relevant documents
#
# Additional MMR parameters (can be added to search_kwargs):
# - fetch_k=20: Number of candidates to consider before applying MMR
# - lambda_mult=0.5: Balance between relevance (1.0) and diversity (0.0)
#   Default is 0.5 (equal balance)

retriever = vectorstore.as_retriever(
    search_type="mmr",  # 🔑 This enables MMR!
    search_kwargs={
        "k": 3,         # Number of documents to return
        # "fetch_k": 20,  # Optional: candidates to consider
        # "lambda_mult": 0.5  # Optional: relevance vs diversity balance
    }
)

print("✅ MMR Retriever created!")
print("=" * 50)
print("📊 Configuration:")
print("   - Search type: MMR (Maximal Marginal Relevance)")
print("   - Documents to return (k): 3")
print("\n💡 MMR will ensure the 3 returned documents are:")
print("   1. Relevant to the query")
print("   2. Diverse from each other (not redundant)")

In [ ]:
# ============================================================================
# STEP 6: CREATE PROMPT TEMPLATE AND INITIALIZE LLM
# ============================================================================
# Set up the prompt and LLM for the RAG pipeline

# Create a simple RAG prompt template
# - {context}: Will be filled with MMR-retrieved documents
# - {input}: Will be filled with the user's question
prompt = PromptTemplate.from_template("""
Answer the question based on the context provided.

Context:
{context}

Question: {input}
""")

# Initialize the LLM using Groq's Gemma2 model
# - gemma2-9b-it: Google's Gemma 2, 9B parameters, instruction-tuned
# - Fast inference via Groq's infrastructure
# - Good at following instructions and answering questions
llm = init_chat_model("groq:gemma2-9b-it")

print("✅ Prompt template and LLM configured!")
print("   - Prompt: Simple context + question format")
print("   - LLM: Gemma2 9B (via Groq)")


In [ ]:
# ============================================================================
# STEP 7: BUILD THE RAG PIPELINE
# ============================================================================
# Combine the MMR retriever with the LLM to create a complete RAG chain

# Create a "stuff" document chain
# "Stuff" strategy: Concatenates all retrieved documents into a single context
# Alternatives: map_reduce, refine, map_rerank
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)

# Create the full RAG chain:
# Query → MMR Retriever → Documents → Prompt + LLM → Answer
rag_chain = create_retrieval_chain(retriever=retriever, combine_docs_chain=document_chain)

print("✅ RAG Pipeline created!")
print("=" * 50)
print("📊 Pipeline Flow:")
print("   1. User query")
print("   2. MMR Retriever (diverse, relevant docs)")
print("   3. Document chain (stuff into context)")
print("   4. LLM generates answer")
print("   5. Return answer + source documents")

In [ ]:
# ============================================================================
# STEP 8: TEST THE MMR RAG PIPELINE
# ============================================================================
# Let's test our pipeline with a question about LangChain features

# Define the query
# This question touches on multiple aspects of LangChain, which is perfect
# for demonstrating MMR's ability to retrieve diverse, relevant content
query = {"input": "How does LangChain support agents and memory?"}

print(f"🔍 Query: \"{query['input']}\"")
print("=" * 60)
print("\n⏳ Running MMR retrieval + LLM generation...")

# Invoke the RAG chain
response = rag_chain.invoke(query)

print("\n" + "=" * 60)
print("✅ ANSWER (Generated by LLM using MMR-retrieved context):")
print("=" * 60)
print(response["answer"])

In [ ]:
# ============================================================================
# STEP 9: EXAMINE THE FULL RESPONSE
# ============================================================================
# The response contains both the answer and the source documents
# Let's examine what MMR retrieved

print("📊 Full Response Object:")
print("=" * 60)
print(f"\n🔑 Keys in response: {list(response.keys())}")

print("\n📄 SOURCE DOCUMENTS (Retrieved by MMR):")
print("-" * 60)
for i, doc in enumerate(response.get("context", []), 1):
    print(f"\n🔹 Document {i}:")
    print(f"   {doc.page_content[:200]}...")

print("\n" + "=" * 60)
print("💡 KEY TAKEAWAYS:")
print("=" * 60)
print("""
1. MMR retrieved documents that are DIVERSE, not just similar
2. Each document covers a different aspect of the query
3. This provides richer context for the LLM to generate answers
4. Without MMR, we might get near-duplicate chunks about the same topic

🎯 MMR is especially useful when:
   - Your document has repetitive content
   - You want comprehensive coverage of a topic
   - Context window space is limited
""")

response